# Phase 6 Groundedness Calibration

Manual review harness for the groundedness judge calibration cases. Run locally with judge credentials configured.

In [ ]:
import json
import os
from datetime import UTC, datetime
from typing import Any

from obs_platform.evaluation.judges.client import AnthropicJudgeClient
from obs_platform.evaluation.judges.calibration import load_all_groundedness_cases
from obs_platform.evaluation.judges.groundedness import GroundednessJudge
from obs_platform.evaluation.types import EvaluationRunView

cases = load_all_groundedness_cases()
len(cases), list(cases)

In [ ]:
def build_run(case_id: str, evidence_texts: list[str], answer_text: str) -> EvaluationRunView:
    timestamp = datetime(2026, 9, 1, 12, 0, tzinfo=UTC)
    return EvaluationRunView(
        run_id=f"calibration-{case_id}",
        schema_version="1.0.0",
        event_type="run_final",
        agent_name="calibration-agent",
        agent_version="manual",
        prompt_version="manual",
        environment="calibration",
        raw_input={"case_id": case_id},
        normalized_input=case_id,
        scenario_id=None,
        started_at=timestamp,
        completed_at=timestamp,
        status="success",
        execution_latency_ms=0,
        wall_clock_duration_ms=0,
        resume_count=0,
        hitl_required=False,
        hitl_state="not_required",
        hitl_checkpoint_id=None,
        hitl_decision=None,
        hitl_requested_at=None,
        hitl_decided_at=None,
        hitl_pending_action=None,
        usage_total_llm_calls=0,
        usage_total_tool_calls=len(evidence_texts),
        usage_total_tokens=0,
        usage_total_retries=0,
        usage_total_estimated_cost_usd=0.0,
        final_result_output={"answer": answer_text},
        final_result_source_references=[],
        runtime_error_category=None,
        runtime_error_code=None,
        runtime_error_message=None,
        runtime_error_failed_component=None,
        spans=[],
        tool_calls=[
            {
                "tool_call_id": f"evidence-{index}",
                "span_id": "calibration-span",
                "tool_name": "calibration_evidence",
                "sequence": index,
                "arguments": {},
                "result": {"text": evidence_text},
                "started_at": timestamp,
                "completed_at": timestamp,
                "latency_ms": 0,
                "retry_count": 0,
                "status": "success",
                "error_category": None,
                "error_code": None,
                "error_message": None,
                "error_failed_component": None,
            }
            for index, evidence_text in enumerate(evidence_texts, start=1)
        ],
        llm_calls=[],
    )


async def run_groundedness_calibration() -> None:
    api_key = os.getenv("JUDGE__ANTHROPIC_API_KEY")
    model = os.getenv("JUDGE__MODEL", "claude-sonnet-4-6")
    if not api_key:
        print("judge credentials not configured, skipping")
        return

    judge = GroundednessJudge(AnthropicJudgeClient(api_key=api_key, model=model))
    for case in cases.values():
        result = await judge.evaluate_async(
            build_run(case.case_id, case.evidence_texts, case.answer_text), []
        )
        print(json.dumps({
            "case_id": case.case_id,
            "expected_label": case.expected_label,
            "passed": result.passed,
            "score": result.score,
            "reason": result.reason,
            "findings": [finding.model_dump() for finding in result.findings],
        }, indent=2))

In [ ]:
await run_groundedness_calibration()